In [2]:
%pip install --quiet --upgrade langchain langchain-core langchain-groq langchain-community langchain-openai pydantic python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import json
from dotenv import load_dotenv
from typing import Optional
from pydantic import BaseModel, Field, field_validator
from typing import Literal
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

load_dotenv()

True

In [4]:
class JobPosting(BaseModel):
    job_title: str
    company: str
    required_skills: list[str]
    experience_level: Literal["Internship", "Junior", "Mid", "Senior", "Lead"]
    salary_min: float = Field(gt=0)
    salary_max: float = Field(gt=0)
    is_remote: bool

    @field_validator("experience_level", mode="before")
    @classmethod
    def normalise_level(cls, v):
        mapping = {
            "intern": "Internship",
            "internship": "Internship",
            "junior": "Junior",
            "entry": "Junior",
            "mid": "Mid",
            "middle": "Mid",
            "senior": "Senior",
            "lead": "Lead",
            "principal": "Lead",
        }
        return mapping.get(v.lower(), v)

In [ ]:
parser = PydanticOutputParser(pydantic_object=JobPosting)

prompt = PromptTemplate(
    template=(
        "Extract the job posting details from the text below.\n\n"
        "{format_instructions}\n\n"
        "Job Description:\n{job_text}"
    ),
    input_variables=["job_text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

Parser and prompt ready.


In [6]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

chain = prompt | llm | parser

In [ ]:
fix_prompt = PromptTemplate(
    template=(
        "The following output was supposed to match this schema but is malformed.\n\n"
        "{format_instructions}\n\n"
        "Malformed output:\n{bad_output}\n\n"
        "Return only the corrected valid JSON."
    ),
    input_variables=["bad_output"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

fix_chain = fix_prompt | llm | parser

Fix chain ready.


In [ ]:
txt_files = [f for f in os.listdir(".") if f.endswith(".txt")]

results = []

for filename in txt_files:
    print(f"Processing: {filename}")

    with open(filename, "r") as f:
        job_text = f.read()

    try:
        job = chain.invoke({"job_text": job_text})
        print(f"  Pass  {filename}")

    except Exception as first_error:
        print(f"  Malformed, retrying with fix prompt for {filename}")
        try:
            bad_output = (prompt | llm).invoke({"job_text": job_text})
            job = fix_chain.invoke({"bad_output": bad_output.content})
            print(f"  Fixed  {filename}")
        except Exception as second_error:
            print(f"  Fail  {filename} - {second_error}")
            continue

    results.append(json.loads(job.model_dump_json()))

print(f"\nDone — {len(results)} jobs extracted out of {len(txt_files)} files")

Processing: job1.txt
  PASS  job1.txt
Processing: job2.txt
  PASS  job2.txt
Processing: job3.txt
  PASS  job3.txt

Done — 3 jobs extracted out of 3 files


In [ ]:
if results:
    r = results[0]
    print("Title:", r["job_title"])
    print("Company:", r["company"])
    print("Skills:", r["required_skills"])
    print("Level:", r["experience_level"])
    print("Salary Range:", r["salary_min"], "-", r["salary_max"])
    print("Remote:", r["is_remote"])

Title        : Senior Backend Engineer
Company      : TechNova Inc.
Skills       : ['Python', 'FastAPI', 'Docker', 'Kubernetes', 'AWS', 'GCP', 'PostgreSQL', 'Redis']
Level        : Senior
Salary Range : 120000.0 - 150000.0
Remote       : True


In [10]:
with open("extracted_jobs.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved {len(results)} records to extracted_jobs.json")

Saved 3 records to extracted_jobs.json
